<a href="https://colab.research.google.com/github/lejonathan24/CSCE676-Project/blob/main/main_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Market Bias in Amazon Electronics Reviews

**Main deliverable notebook for CSCE 676.**

This project investigates whether product presentation, especially the gender attribute of product models, is associated with systematic differences in product ratings in the Market Bias Electronics dataset. The notebook is curated around the final project story: motivation, data, research questions, methods, results, and conclusions.


## Motivation

Online product reviews are often treated as direct signals of product quality, but ratings may also reflect presentation choices and market bias. In the Electronics portion of the Market Bias dataset, products include attributes such as category, brand, rating, and `model_attr`, which represents the gender presentation of the product model. This project asks whether those presentation attributes are merely descriptive or whether they relate to measurable rating differences.

I focus on the Electronics dataset because it is large enough to support meaningful pattern mining and statistical modeling, but still practical to analyze in Colab without distributed computing. The tradeoff is that the analysis has to manage sparse brands, missing values, and highly skewed ratings.


## Research Questions

**RQ1 — Pattern Mining:** Do certain combinations of product attributes and model gender frequently co-occur with high or low ratings?

**RQ2 — Clustering:** Do distinct clusters of products emerge based on product attributes and ratings, and do those clusters reflect systematic model-gender differences?

**RQ3 — Causal/Fairness Analysis:** Does model gender have a measurable relationship with product ratings after controlling for category and brand?

These questions align with the assignment guidance from the earlier checkpoints: RQ1 uses frequent itemset/association rule mining, RQ2 uses clustering, and RQ3 uses regression-based causal/fairness analysis as an external method.


## Setup and Data Loading

Place the dataset at `data/df_electronics.csv`, or run in Colab and update the Google Drive path if needed.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Dataset path used by this repository.
DATA_PATH = Path("data/df_electronics.csv")

# If running in Colab and the data file is not in the repo, mount Google Drive
# and update DATA_PATH to the exact file location if needed.
if not DATA_PATH.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        # Example fallback location. Change this if your Drive folder is different.
        DATA_PATH = Path("/content/drive/MyDrive/CSCE676 Project Data/df_electronics.csv")
    except Exception:
        pass

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Place the file at data/df_electronics.csv "
        "or update DATA_PATH to the correct location."
    )

df_elec = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Electronics shape:", df_elec.shape)
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="jupyter_client.session"
)


## Data Preparation and Descriptive Summary

The analysis keeps the three model-gender groups used throughout the project: `Male`, `Female`, and `Female&Male`. Ratings are also converted into interpretable buckets for pattern mining. I use rating buckets rather than raw rating values because association rules are easier to interpret as high-, medium-, and low-rating outcomes.


In [ ]:
import pandas as pd

# Copy and keep only relevant model groups
df = df_elec.copy()
df = df[df["model_attr"].isin(["Male", "Female", "Female&Male"])].copy()

# -----------------------------
# 1. Dataset size
# -----------------------------
n_reviews = len(df)
print(f"Amazon Electronics Dataset (~{n_reviews:,} reviews)")

# -----------------------------
# 2. Rating distribution
# -----------------------------
rating_counts = (
    df["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)

rating_counts["percent"] = rating_counts["count"] / rating_counts["count"].sum()

print("\nRating Distribution:")
print(rating_counts)

# -----------------------------
# 3. Mean rating by model gender
# -----------------------------
mean_rating_by_gender = (
    df.groupby("model_attr")["rating"]
    .mean()
    .reindex(["Male", "Female", "Female&Male"])
    .reset_index(name="mean_rating")
)

print("\nMean Rating by Model Gender:")
print(mean_rating_by_gender)

# -----------------------------
# 4. 5-star rate by model gender
# -----------------------------
five_star_rate_by_gender = (
    df.assign(is_5_star=(df["rating"] == 5).astype(int))
    .groupby("model_attr")["is_5_star"]
    .mean()
    .reindex(["Male", "Female", "Female&Male"])
    .reset_index(name="five_star_rate")
)

print("\n5-Star Rate by Model Gender:")
print(five_star_rate_by_gender)

# -----------------------------
# 5. Optional: combined summary table
# -----------------------------
slide_summary = mean_rating_by_gender.merge(
    five_star_rate_by_gender,
    on="model_attr"
)

print("\nSlide Summary:")
print(slide_summary)

In [ ]:
import matplotlib.pyplot as plt

# -----------------------------
# Graph 1: Rating distribution
# -----------------------------
plt.figure(figsize=(7,5))
plt.bar(rating_counts["rating"].astype(str), rating_counts["count"])
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# -----------------------------
# Graph 2: Mean rating by model gender
# -----------------------------
plt.figure(figsize=(7,5))
plt.bar(mean_rating_by_gender["model_attr"], mean_rating_by_gender["mean_rating"])
plt.title("Average Rating by Model Gender")
plt.xlabel("Model Gender")
plt.ylabel("Average Rating")
plt.tight_layout()
plt.show()

# -----------------------------
# Graph 3: 5-star rate by model gender
# -----------------------------
plt.figure(figsize=(7,5))
plt.bar(five_star_rate_by_gender["model_attr"], five_star_rate_by_gender["five_star_rate"])
plt.title("5-Star Rate by Model Gender")
plt.xlabel("Model Gender")
plt.ylabel("Proportion of 5-Star Ratings")
plt.tight_layout()
plt.show()

## RQ1: Association Rules and FP-Growth

For RQ1, I use FP-Growth because it is more scalable than a naive Apriori-style search on a dataset with many repeated transactions. To keep the rules readable and avoid brand sparsity, I restrict the pattern-mining table to the top brands. The goal is not to claim that gender alone determines ratings, but to see whether model gender appears inside attribute combinations that are associated with high or low rating outcomes.

The rating buckets are:
- `Low`: ratings 1–3
- `Medium`: rating 4
- `High`: rating 5


In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

df_pm = df[["model_attr", "category", "brand", "rating"]].dropna().copy()

# Keep top brands so the rules are readable and not too sparse
top_brands = df_pm["brand"].value_counts().head(25).index
df_pm = df_pm[df_pm["brand"].isin(top_brands)].copy()

# Create rating buckets
df_pm["rating_label"] = pd.cut(
    df_pm["rating"],
    bins=[0, 3, 4, 5],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

print(df_pm["rating_label"].value_counts())
print(df_pm.shape)

In [ ]:
# Sample for speed if needed
sample_n = min(120000, len(df_pm))
df_pm_sample = df_pm.sample(n=sample_n, random_state=42).copy()

transactions = df_pm_sample[["model_attr", "category", "brand", "rating_label"]].astype(str).values.tolist()

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_array, columns=te.columns_)

print(df_trans.shape)
df_trans.head()

In [ ]:
freq_items = fpgrowth(
    df_trans,
    min_support=0.01,
    use_colnames=True
)

rules = association_rules(
    freq_items,
    metric="lift",
    min_threshold=1.0
)

# Keep only rules whose consequence is rating-related
rules_rating = rules[
    rules["consequents"].apply(lambda x: any(label in x for label in ["High", "Medium", "Low"]))
].copy()

rules_rating = rules_rating.sort_values(["lift", "confidence"], ascending=False)

print(rules_rating[["antecedents", "consequents", "support", "confidence", "lift"]].head(20))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
rules_high = rules_rating[
    rules_rating["consequents"].apply(lambda x: "High" in x)
].copy()

rules_high_gender = rules_high[
    rules_high["antecedents"].apply(lambda x: any(g in x for g in ["Male", "Female", "Female&Male"]))
].sort_values(["lift", "confidence"], ascending=False)

print("Top gender-related rules predicting HIGH ratings:")
print(rules_high_gender[["antecedents", "consequents", "support", "confidence", "lift"]].head(15))

In [ ]:
rules_low = rules_rating[
    rules_rating["consequents"].apply(lambda x: "Low" in x)
].copy()

rules_low_gender = rules_low[
    rules_low["antecedents"].apply(lambda x: any(g in x for g in ["Male", "Female", "Female&Male"]))
].sort_values(["lift", "confidence"], ascending=False)

print("Top gender-related rules predicting LOW ratings:")
print(rules_low_gender[["antecedents", "consequents", "support", "confidence", "lift"]].head(15))

In [ ]:
pm_summary = pd.crosstab(df_pm["model_attr"], df_pm["rating_label"], normalize="index")
print(pm_summary)

pm_summary.plot(kind="bar", figsize=(8,5))
plt.title("Rating Bucket Distribution by Model Gender")
plt.xlabel("Model Gender")
plt.ylabel("Proportion")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Deeper RQ1 Relationship Analysis: Rating Outcomes by Model Gender

The FP-Growth rules show which item combinations appear together, but rules alone can be misleading because high-support groups may dominate the results. To make the relationship clearer, this section compares each model-gender group against the overall baseline rating behavior. I use both raw proportions and lift-style ratios so the findings are easier to interpret.


In [ ]:
# Relationship table: model gender vs rating bucket outcomes
relationship_table = (
    df_pm.groupby("model_attr")
    .agg(
        n_reviews=("rating", "size"),
        mean_rating=("rating", "mean"),
        high_rate=("rating_label", lambda s: (s == "High").mean()),
        medium_rate=("rating_label", lambda s: (s == "Medium").mean()),
        low_rate=("rating_label", lambda s: (s == "Low").mean()),
    )
    .sort_values("mean_rating", ascending=False)
)

overall_high_rate = (df_pm["rating_label"] == "High").mean()
overall_low_rate = (df_pm["rating_label"] == "Low").mean()
relationship_table["high_lift_vs_dataset"] = relationship_table["high_rate"] / overall_high_rate
relationship_table["low_lift_vs_dataset"] = relationship_table["low_rate"] / overall_low_rate

print("Overall high-rating rate:", round(overall_high_rate, 4))
print("Overall low-rating rate:", round(overall_low_rate, 4))
display(relationship_table.round(4))


In [ ]:
# Visualize the high/low outcome relationship in one place
plot_cols = ["high_rate", "medium_rate", "low_rate"]
relationship_table[plot_cols].plot(kind="bar", figsize=(9, 5))
plt.title("Rating Outcome Rates by Model Gender")
plt.xlabel("Model Gender")
plt.ylabel("Proportion of Reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Category-Level Relationship Check

A major risk in this project is confusing a gender relationship with a category relationship. For example, if one model-gender group appears more often in categories that already tend to receive higher ratings, the raw gender gap could be misleading. The next table checks the gender-rating relationship separately within product categories and highlights where the largest differences appear.


In [ ]:
# Category-level model gender gaps
cat_gender_summary = (
    df_pm.groupby(["category", "model_attr"])
    .agg(
        n_reviews=("rating", "size"),
        mean_rating=("rating", "mean"),
        high_rate=("rating_label", lambda s: (s == "High").mean()),
        low_rate=("rating_label", lambda s: (s == "Low").mean()),
    )
    .reset_index()
)

# Only keep category/gender cells with enough observations to avoid unstable comparisons
min_cell_size = 50
cat_gender_summary = cat_gender_summary[cat_gender_summary["n_reviews"] >= min_cell_size].copy()

mean_pivot = cat_gender_summary.pivot(index="category", columns="model_attr", values="mean_rating")
high_pivot = cat_gender_summary.pivot(index="category", columns="model_attr", values="high_rate")

# Compute interpretable gaps where both groups are present
for comparison, left, right in [
    ("Female_minus_Male", "Female", "Male"),
    ("Female&Male_minus_Male", "Female&Male", "Male"),
]:
    if left in mean_pivot.columns and right in mean_pivot.columns:
        mean_pivot[comparison] = mean_pivot[left] - mean_pivot[right]
    if left in high_pivot.columns and right in high_pivot.columns:
        high_pivot[comparison] = high_pivot[left] - high_pivot[right]

print("Largest category-level mean rating gaps:")
display(mean_pivot.filter(like="minus").dropna(how="all").sort_values(
    by=mean_pivot.filter(like="minus").columns.tolist()[0],
    key=lambda s: s.abs(),
    ascending=False
).head(10).round(4))

print("Largest category-level high-rating-rate gaps:")
display(high_pivot.filter(like="minus").dropna(how="all").sort_values(
    by=high_pivot.filter(like="minus").columns.tolist()[0],
    key=lambda s: s.abs(),
    ascending=False
).head(10).round(4))


### FP-Growth Sensitivity Check

The number and type of patterns found by FP-Growth depends heavily on the minimum support threshold. This directly connects to the project write-up: lower support finds more specific patterns, but it can also introduce noisier or less reliable rules. Higher support gives fewer, broader rules that are easier to trust but may miss smaller subgroup effects.


In [ ]:
# Sensitivity analysis for support threshold choices
support_values = [0.005, 0.01, 0.02, 0.05]
sensitivity_rows = []

for support in support_values:
    freq_tmp = fpgrowth(df_trans, min_support=support, use_colnames=True)
    if len(freq_tmp) == 0:
        sensitivity_rows.append({
            "min_support": support,
            "n_itemsets": 0,
            "n_rating_rules": 0,
            "n_gender_rating_rules": 0,
            "max_lift_gender_rating_rule": np.nan,
        })
        continue

    rules_tmp = association_rules(freq_tmp, metric="lift", min_threshold=1.0)
    rating_rules_tmp = rules_tmp[
        rules_tmp["consequents"].apply(lambda x: any(label in x for label in ["High", "Medium", "Low"]))
    ].copy()
    gender_rating_rules_tmp = rating_rules_tmp[
        rating_rules_tmp["antecedents"].apply(lambda x: any(g in x for g in ["Male", "Female", "Female&Male"]))
    ].copy()
    sensitivity_rows.append({
        "min_support": support,
        "n_itemsets": len(freq_tmp),
        "n_rating_rules": len(rating_rules_tmp),
        "n_gender_rating_rules": len(gender_rating_rules_tmp),
        "max_lift_gender_rating_rule": gender_rating_rules_tmp["lift"].max() if len(gender_rating_rules_tmp) else np.nan,
    })

support_sensitivity = pd.DataFrame(sensitivity_rows)
display(support_sensitivity)

support_sensitivity.plot(x="min_support", y=["n_itemsets", "n_rating_rules", "n_gender_rating_rules"], marker="o", figsize=(8, 5))
plt.title("FP-Growth Sensitivity to Minimum Support")
plt.xlabel("Minimum Support")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


# Pattern Mining Results
Pattern mining reveals that model gender appears in combinations of product attributes associated with rating outcomes, rather than acting independently. Female and mixed-gender products frequently occur in high-rating association rules, often alongside specific brands and categories, while male-presented products appear more often in rules linked to lower ratings. Importantly, these patterns are context-dependent, certain male-presented products still appear in high-rating combinations, indicating that the effect of model gender is not uniform but interacts with other product attributes. Overall, this suggests that model gender contributes to rating differences through its role within broader attribute combinations, providing evidence of subtle but systematic bias in how products are evaluated.


### RQ1 Interpretation

The most important pattern-mining takeaway is that gender-related rules should be interpreted contextually. A frequent itemset can be misleading if it appears to suggest a gender effect but is actually driven by a dominant brand or category. Because of this, I compare support, confidence, and lift rather than relying on raw frequency alone. As support thresholds become stricter, the number of discovered patterns drops, but the remaining rules become easier to interpret and less noisy.


## RQ2: Product Clustering

For RQ2, I cluster products using category, brand, model gender, and rating. Categorical fields are one-hot encoded and rating is standardized. I evaluate cluster quality with silhouette score and then inspect rating means, model-gender distributions, category distributions, and top brands within each cluster.


In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

df_cl = df[["category", "brand", "model_attr", "rating"]].dropna().copy()

# Keep top brands for a cleaner clustering space
top_brands = df_cl["brand"].value_counts().head(25).index
df_cl = df_cl[df_cl["brand"].isin(top_brands)].copy()

# Sample for speed
sample_n = min(15000, len(df_cl))
df_cl = df_cl.sample(n=sample_n, random_state=42).copy()

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_cat = encoder.fit_transform(df_cl[["category", "brand", "model_attr"]])

scaler = StandardScaler()
X_num = scaler.fit_transform(df_cl[["rating"]])

X = np.hstack([X_cat, X_num])

print(X.shape)

In [ ]:
sil_scores = []
k_values = range(2, 9)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    sil_scores.append((k, sil))

sil_df = pd.DataFrame(sil_scores, columns=["k", "silhouette"])
print(sil_df)

plt.figure(figsize=(7,4))
plt.plot(sil_df["k"], sil_df["silhouette"], marker="o")
plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

In [ ]:
best_k = 2

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cl["cluster"] = kmeans.fit_predict(X)

print("Final silhouette:", silhouette_score(X, df_cl["cluster"]))

In [ ]:
print(df_cl["cluster"].value_counts().sort_index())

In [ ]:
cluster_rating = df_cl.groupby("cluster")["rating"].mean().sort_values()
print(cluster_rating)

cluster_rating.plot(kind="bar", figsize=(7,4))
plt.title("Average Rating by Cluster")
plt.xlabel("Cluster")
plt.ylabel("Average Rating")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
cluster_gender = pd.crosstab(df_cl["cluster"], df_cl["model_attr"], normalize="index")
print(cluster_gender)

cluster_gender.plot(kind="bar", stacked=True, figsize=(8,5))
plt.title("Model Gender Distribution Within Clusters")
plt.xlabel("Cluster")
plt.ylabel("Proportion")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
cluster_category = pd.crosstab(df_cl["cluster"], df_cl["category"], normalize="index")
print(cluster_category.round(3))

In [ ]:
for c in sorted(df_cl["cluster"].unique()):
    print(f"\nTop brands in cluster {c}:")
    print(df_cl[df_cl["cluster"] == c]["brand"].value_counts().head(10))

In [ ]:
# Without model_attr
X_cat_no_gender = encoder.fit_transform(df_cl[["category", "brand"]])
X_no_gender = np.hstack([X_cat_no_gender, X_num])

sil_scores_no_gender = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_no_gender)
    sil = silhouette_score(X_no_gender, labels)
    sil_scores_no_gender.append((k, sil))

sil_no_gender_df = pd.DataFrame(sil_scores_no_gender, columns=["k", "silhouette_no_gender"])
compare_sil = sil_df.merge(sil_no_gender_df, on="k")
print(compare_sil)

### Deeper RQ2 Cluster Interpretation

The clustering section should not only report that clusters exist; it should explain what makes each cluster different. The profile below summarizes each cluster by size, average rating, model-gender mix, and dominant product categories/brands. This makes the cluster results easier to connect back to the project question.


In [ ]:
# Compact cluster profiles
cluster_profiles = []
for cluster_id, part in df_cl.groupby("cluster"):
    top_gender = part["model_attr"].value_counts(normalize=True).head(3)
    top_categories = part["category"].value_counts(normalize=True).head(3)
    top_brands = part["brand"].value_counts(normalize=True).head(3)
    cluster_profiles.append({
        "cluster": cluster_id,
        "n_reviews": len(part),
        "mean_rating": part["rating"].mean(),
        "top_model_gender_mix": "; ".join([f"{idx}: {val:.1%}" for idx, val in top_gender.items()]),
        "top_categories": "; ".join([f"{idx}: {val:.1%}" for idx, val in top_categories.items()]),
        "top_brands": "; ".join([f"{idx}: {val:.1%}" for idx, val in top_brands.items()]),
    })

cluster_profiles_df = pd.DataFrame(cluster_profiles).sort_values("cluster")
display(cluster_profiles_df)


In [ ]:
# Compare whether clusters separate rating behavior beyond model gender alone
cluster_gender_rating = (
    df_cl.groupby(["cluster", "model_attr"])
    .agg(n_reviews=("rating", "size"), mean_rating=("rating", "mean"))
    .reset_index()
    .sort_values(["cluster", "mean_rating"], ascending=[True, False])
)

display(cluster_gender_rating.round(4))


# Clustering Results
Clustering reveals two main groups of products: high-rated and low-rated. However, these clusters are not strongly driven by category or brand, and including model gender does not significantly improve clustering structure. This suggests that while model gender influences ratings, it does so as a subtle effect rather than defining distinct product groups.


### RQ2 Interpretation

The clustering results are useful mostly as a contrast to RQ1 and RQ3. The clusters separate high-rated and lower-rated products more clearly than they separate gender groups. This suggests that model gender is not the main structural axis of the dataset, even though it still appears in association rules and regression results.


## RQ3: Controlled Regression and Fairness-Oriented Analysis

For RQ3, I estimate the relationship between model gender and rating while controlling for category and brand. The baseline group is `Male`, so the coefficients for `Female` and `Female&Male` can be read as average rating differences relative to male-presented products, after accounting for observable product category and brand differences.

This is not a full randomized causal experiment, so I avoid claiming absolute causality. Instead, I interpret the results as evidence of a controlled association that remains after including major product controls.


In [ ]:
import statsmodels.formula.api as smf

df_ca = df[["rating", "model_attr", "category", "brand"]].dropna().copy()

# Keep top brands for stability and interpretability
top_brands = df_ca["brand"].value_counts().head(50).index
df_ca = df_ca[df_ca["brand"].isin(top_brands)].copy()

# Set Male as reference group
df_ca["model_attr"] = pd.Categorical(
    df_ca["model_attr"],
    categories=["Male", "Female", "Female&Male"],
    ordered=False
)

print(df_ca.shape)
print(df_ca["model_attr"].value_counts())

In [ ]:
ols_model = smf.ols(
    "rating ~ C(model_attr, Treatment(reference='Male')) + C(category) + C(brand)",
    data=df_ca
).fit()

print(ols_model.summary())

In [ ]:
coef_table = ols_model.summary2().tables[1]

rows_of_interest = [
    "C(model_attr, Treatment(reference='Male'))[T.Female]",
    "C(model_attr, Treatment(reference='Male'))[T.Female&Male]"
]

print(coef_table.loc[rows_of_interest, ["Coef.", "Std.Err.", "t", "P>|t|", "[0.025", "0.975]"]])

In [ ]:
raw_means = df_ca.groupby("model_attr")["rating"].mean()

raw_gap_female = raw_means["Female"] - raw_means["Male"]
raw_gap_both = raw_means["Female&Male"] - raw_means["Male"]

controlled_female = coef_table.loc[
    "C(model_attr, Treatment(reference='Male'))[T.Female]", "Coef."
]
controlled_both = coef_table.loc[
    "C(model_attr, Treatment(reference='Male'))[T.Female&Male]", "Coef."
]

print(f"Raw Female - Male gap: {raw_gap_female:.4f}")
print(f"Raw Female&Male - Male gap: {raw_gap_both:.4f}")
print(f"Controlled Female - Male effect: {controlled_female:.4f}")
print(f"Controlled Female&Male - Male effect: {controlled_both:.4f}")

In [ ]:
df_log = df_ca.copy()
df_log["high_rating"] = (df_log["rating"] >= 5).astype(int)

logit_model = smf.logit(
    "high_rating ~ C(model_attr, Treatment(reference='Male')) + C(category) + C(brand)",
    data=df_log
).fit(maxiter=200)

print(logit_model.summary())

In [ ]:
logit_params = logit_model.params
odds_ratios = np.exp(logit_params)

or_rows = [
    "C(model_attr, Treatment(reference='Male'))[T.Female]",
    "C(model_attr, Treatment(reference='Male'))[T.Female&Male]"
]

print("Odds Ratios for 5-star outcome:")
print(odds_ratios.loc[or_rows])

In [ ]:
coef_plot = coef_table.loc[rows_of_interest, ["Coef.", "[0.025", "0.975]"]].copy()
coef_plot["label"] = ["Female vs Male", "Female&Male vs Male"]

plt.figure(figsize=(7,4))
plt.errorbar(
    coef_plot["Coef."],
    coef_plot["label"],
    xerr=[
        coef_plot["Coef."] - coef_plot["[0.025"],
        coef_plot["0.975]"] - coef_plot["Coef."]
    ],
    fmt="o"
)
plt.axvline(0, linestyle="--")
plt.title("Controlled Effect of Model Gender on Rating")
plt.xlabel("Coefficient Estimate")
plt.tight_layout()
plt.show()

### Deeper RQ3: Interaction Analysis

The controlled regression estimates the average relationship between model gender and rating after accounting for product category and brand. However, an average effect can hide category-specific differences. This interaction model asks whether the gender-rating relationship changes across categories. If interaction terms are large or significant, it suggests the relationship is not uniform across the marketplace.


In [ ]:
# Interaction model: does the model-gender relationship vary by category?
interaction_model = smf.ols(
    "rating ~ C(model_attr, Treatment(reference='Male')) * C(category) + C(brand)",
    data=df_ca
).fit()

interaction_terms = interaction_model.summary2().tables[1]
interaction_terms = interaction_terms[interaction_terms.index.str.contains(":C\(category\)", regex=True)].copy()
interaction_terms["abs_coef"] = interaction_terms["Coef."].abs()

print("Largest model-gender by category interaction terms:")
display(interaction_terms.sort_values("abs_coef", ascending=False)[["Coef.", "Std.Err.", "P>|t|", "[0.025", "0.975]"]].head(15).round(4))


### Practical Significance Check

Statistical significance can be misleading in large datasets because even tiny effects may become significant. This section compares raw and controlled gaps to the full 1–5 rating scale, so the conclusion discusses practical size, not just p-values.


In [ ]:
# Practical effect size relative to the 1-5 star scale
rating_scale_width = df_ca["rating"].max() - df_ca["rating"].min()
effect_size_summary = pd.DataFrame({
    "comparison": ["Female - Male", "Female&Male - Male"],
    "raw_gap_stars": [raw_gap_female, raw_gap_both],
    "controlled_gap_stars": [controlled_female, controlled_both],
})
effect_size_summary["controlled_gap_pct_of_scale"] = (
    effect_size_summary["controlled_gap_stars"].abs() / rating_scale_width * 100
)

display(effect_size_summary.round(4))


# Causal Analysis Results

Causal analysis shows that the effect of model gender on product ratings persists even after controlling for category and brand. Compared to male-presented products, products with female models receive an average rating increase of approximately 0.083 points, while products featuring both male and female models receive an increase of approximately 0.135 points, both of which are highly statistically significant (p < 0.001). Notably, these effects remain after accounting for product type and brand differences, indicating that they cannot be explained solely by observable product characteristics. A logistic regression further confirms these findings, showing that female-presented products have approximately 10.6% higher odds of receiving a 5-star rating, and mixed-gender products have approximately 17.4% higher odds. Together, these results provide strong evidence that model gender has a measurable and systematic influence on product ratings.


## Final Takeaways

This notebook examines whether model-gender presentation in Amazon Electronics reviews is related to rating outcomes. The analysis uses three complementary views:

1. **Pattern mining / FP-Growth** identifies recurring product, brand, category, model-gender, and rating combinations. The support sensitivity check shows how stricter support thresholds produce fewer but more stable patterns, while lower thresholds reveal more specific subgroup relationships.
2. **Clustering** checks whether reviews naturally group into segments with different rating behavior and model-gender composition. The cluster profiles help interpret whether clusters are mainly driven by product/category structure or by model-gender patterns.
3. **Controlled regression** compares raw rating gaps against estimates that account for category and brand. This is important because a raw difference can be caused by uneven representation across categories or brands rather than model gender itself.

The main conclusion should be interpreted carefully: the project can identify **associations and potential subgroup patterns**, but it should not claim true causal bias without stronger experimental or causal identification. The strongest evidence comes from relationships that remain visible after category/brand controls and that are consistent across pattern mining, clustering, and regression.


# **On my honor, I declare the following resources:**
1. Collaborators:
-

2. Web Sources:
- Homework 1 & 2 for EDA inspiration
- https://www.icwsm.org/2025/schedule/allpapers.html
- https://kdd2025.kdd.org/datasets-and-benchmarks-track-papers-2/
- https://pandas.pydata.org/?utm_source=chatgpt.com – for data cleaning and transformation
- https://scikit-learn.org/?utm_source=chatgpt.com – for clustering, preprocessing, and evaluation metrics
- https://www.statsmodels.org/?utm_source=chatgpt.com – for OLS regression setup and interpretation
- https://seaborn.pydata.org/?utm_source=chatgpt.com – for exploratory data analysis and plotting
- https://matplotlib.org/?utm_source=chatgpt.com – for visualizations used during EDA


3. AI Tools:
- chatGPT: Used to compare the three datasets for the Comparative Analysis of Datasets
- chatGPT: Used to help generate a EDA
- chatGPT: Provided with EDA results and asked to give observations, hypothesis, and potential RQs
- chatGPT: Used to debug code errors (e.g., OneHotEncoder parameter issues, model setup)

- chatGPT: Used to explain clustering evaluation metrics (e.g., silhouette score)

- chatGPT: Used to refine explanations of algorithmic decisions and interpretations

- chatGPT: Used to structure and polish written responses for clarity


## Environment Export

Run the cell below in Colab to export the exact package versions from your session.

```python
!pip freeze > requirements.txt
from google.colab import files
files.download('requirements.txt')
!python --version
```
